# LISS-IV product-folder sorting by Path/Row (2026)

This notebook scans local product folders recursively, reads Path, Row, and DateOfPass from product metadata, groups matching 2026 pre/post folders, and can move each complete product directory to a same-drive output folder. It keeps all bands, metadata, and sidecars together. This step does not require georeferencing; the PCA app checks CRS and pixel-grid compatibility later.

**Move safety:** review the inventory and preview first. `CONFIRM_MOVE` starts as `False`; set it to `True` only when you are ready to move the selected folders. Moves stay on one drive, existing product destinations are never overwritten, and unmatched folders remain in place.


In [ ]:
from pathlib import Path
from IPython.display import display
import bulk_scene_prep as bulk

# Edit these paths. A root can contain many nested product folders.
SOURCE_ROOTS = [Path(r"F:\SILAS\LISS4_2026")]
OUTPUT_ROOT = Path(r"F:\SILAS\LISS4_Sorted_By_Path_Row_2026")

# None selects every matched Path/Row group. Or choose specific pairs, for example [(94, 45)].
SELECTED_PATH_ROWS = None

# Leave False until you have reviewed the groups and move preview below.
CONFIRM_MOVE = False


In [ ]:
# Scan and deduplicate products if source roots overlap.
scenes_by_folder = {}
for source_root in SOURCE_ROOTS:
    if not source_root.is_dir():
        raise NotADirectoryError(f"Source folder does not exist: {source_root}")
    for scene in bulk.scan_root(source_root, ignore_root=OUTPUT_ROOT):
        scenes_by_folder[str(Path(scene["folder"]).resolve()).casefold()] = scene

SCENES = list(scenes_by_folder.values())
GROUPS, CANDIDATES, SKIPPED = bulk.group_pathrow_products(SCENES)

print(f"Complete products in the 2026 date windows: {len(CANDIDATES)}")
print(f"Matched Path/Row groups: {len(GROUPS)}")
print(f"Skipped products with missing metadata or unclear B2/B3/B4: {len(SKIPPED)}")

display([
    {
        "Path": group["path"],
        "Row": group["row"],
        "Pre products": len(group["pre"]),
        "Pre dates": ", ".join(scene["date"] for scene in group["pre"]),
        "Post products": len(group["post"]),
        "Post dates": ", ".join(scene["date"] for scene in group["post"]),
    }
    for group in GROUPS
])

display([
    {
        "Path": scene["path"],
        "Row": scene["row"],
        "Season": scene["season"],
        "Date": scene["date"],
        "Bands": scene["band_layout"],
        "Product folder": scene["folder"],
    }
    for scene in CANDIDATES
])
if SKIPPED:
    display(SKIPPED[:50])


In [ ]:
# Choose all matches or a subset using SELECTED_PATH_ROWS in the configuration cell.
AVAILABLE_PATH_ROWS = {(group["path"], group["row"]) for group in GROUPS}
if SELECTED_PATH_ROWS is None:
    SELECTED_KEYS = AVAILABLE_PATH_ROWS
else:
    SELECTED_KEYS = {tuple(map(int, key)) for key in SELECTED_PATH_ROWS}
    unknown = SELECTED_KEYS - AVAILABLE_PATH_ROWS
    if unknown:
        raise ValueError(f"Selected Path/Row values were not found: {sorted(unknown)}")

SELECTED_GROUPS = [
    group for group in GROUPS if (group["path"], group["row"]) in SELECTED_KEYS
]
MOVE_PREVIEW = []
for group in SELECTED_GROUPS:
    pathrow_folder = OUTPUT_ROOT / f"Path_{group['path']:04d}_Row_{group['row']:04d}"
    for season in ("pre", "post"):
        season_folder = "Pre_Monsoon" if season == "pre" else "Post_Monsoon"
        for scene in group[season]:
            MOVE_PREVIEW.append({
                "Path": group["path"],
                "Row": group["row"],
                "Season": season_folder,
                "Date": scene["date"],
                "Source folder to move": scene["folder"],
                "New Path/Row/season folder": str(pathrow_folder / season_folder),
            })

display(MOVE_PREVIEW)
print(f"Selected {len(MOVE_PREVIEW)} complete product folders.")


In [ ]:
if not CONFIRM_MOVE:
    print("Nothing moved. Review the tables above. When ready, set CONFIRM_MOVE = True in the configuration cell and run this cell again.")
else:
    MOVED_FOLDERS = bulk.move_pathrow_products(
        SELECTED_GROUPS,
        SELECTED_KEYS,
        OUTPUT_ROOT,
        SOURCE_ROOTS,
    )
    print(f"Moved {len(MOVED_FOLDERS)} complete product folders.")
    print(f"Index CSV: {OUTPUT_ROOT / 'path_row_folder_index.csv'}")
    display(MOVED_FOLDERS)


## Next: validate and run PCA

For a matched Path/Row, copy the exact product-folder paths from the new `Pre_Monsoon` and `Post_Monsoon` folders into the local Landslide PCA browser app. The app reads BAND2, BAND3, and BAND4 from each folder in order, checks metadata Path/Row plus the georeferencing and pixel grid of all six rasters, creates the SOP stack, and writes PCA results under the same Path/Row folder by default. Complete image co-registration before PCA; matching georeferencing metadata alone does not prove sub-pixel alignment.
